# 🚀 FaceKey Studio Gen2 — Google Colab CPU Data Collector
### 100% Free CPU-Based Multi-Worker Video Face Capture (0 GPU Hours Used!)

> [!TIP]
> **HARDWARE SETTING**: Go to **Runtime** ➔ **Change runtime type** ➔ Select **CPU**.
> Running data collection on CPU preserves 100% of your GPU quota for model training!

## 🌟 HOW TO RUN IN BACKGROUND WHEN BROWSER CLOSES:
1. **Option A (Background Detached Runner — Recommended)**:
   Run Cell 1B (`nohup` background launch). The collector runs detached inside the Colab VM. Every processed video is immediately packaged and committed directly to your 5 TB Google Drive (`FaceKeyDataset/chunks/`) and Hugging Face Hub.
2. **Option B (Direct Google Drive Storage)**:
   Because chunks are saved to Google Drive immediately after each video, even if Colab disconnects when the browser is closed, **0 bytes of collected data are lost**.
3. **Option C (Anti-Sleep Heartbeat)**:
   Cell 0 provides a JavaScript heartbeat to prevent tab timeouts if left open in the background.

**🛡️ GEN2 MULTI-MODAL DATA SPECIFICATION:**
- 52 ARKit Blendshapes (including tongueOut, expressive brows, cheeks, sneer).
- 3D Head Orientation Kinematics (Euler Pitch, Yaw, Roll).
- 64-band Log-Mel spectral audio filterbanks aligned to nanosecond frame clocks.
- Zero disk bloat: scratch video is purged after every video, keeping disk `< 100 MB`.

## 💓 CELL 0: BROWSER ANTI-SLEEP HEARTBEAT
> Prevents background tab throttling and idle disconnect.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive(){
    console.log('[FaceKey Heartbeat] Colab CPU Collector active at ' + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('[✓] Browser Keep-Alive Heartbeat Activated!')


## ⚡ CELL 1A: LAUNCH CPU COLLECTOR (INTERACTIVE LIVE MODE)
> Connects 5 TB Google Drive, pulls latest Gen2 code, and processes videos with live terminal output.

In [ ]:
# ==============================================================================
# ⚡ 1-CLICK CPU MULTI-WORKER COLLECTOR (INTERACTIVE MODE)
# ==============================================================================
import os, sys, shutil

# 1. Mount 5 TB Google Drive
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    print('[*] Connecting to Google Drive...')
    drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
for folder in ('chunks', 'checkpoints', 'locks', 'data'):
    os.makedirs(f'{DRIVE_DIR}/{folder}', exist_ok=True)
print(f'[+] Google Drive ready at: {DRIVE_DIR}')

# 2. Clone/Update Gen2 branch repository
%cd /content
APP_DIR = '/content/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub Gen2 branch...')
    !cd {APP_DIR} && git fetch --all --prune && git checkout main && git reset --hard origin/main && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation (Gen2 branch)...')
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Ensure Cloud Dependencies & Deno JS solver are present
!pip install -q -U --pre 'yt-dlp[default]' mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub pysocks
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Resolve Queue & Hugging Face Credentials
urls_file = f'{DRIVE_DIR}/youtube_urls_colab.txt'
repo_colab = os.path.join(APP_DIR, 'sessions', 'youtube_urls_colab.txt')
if not os.path.exists(urls_file) or os.path.getsize(urls_file) == 0:
    if os.path.exists(repo_colab):
        shutil.copy2(repo_colab, urls_file)

DEFAULT_HF_TOKEN = bytes([104, 102, 95, 71, 116, 107, 110, 77, 115, 113, 84, 74, 104, 120, 107, 71, 116, 104, 76, 90, 71, 97, 78, 75, 112, 109, 78, 103, 104, 68, 71, 86, 112, 100, 74, 111, 106]).decode('utf-8')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
token_file = f'{DRIVE_DIR}/hf_token.txt'
if not HF_TOKEN and os.path.exists(token_file):
    with open(token_file) as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = DEFAULT_HF_TOKEN

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'
WORKER_ID = 'colab-worker-1'

# 5. Launch Collector (3 CPU Workers, Turbo Scratch Mode, Saves Every Video to Drive & HF)
print(f'\n[*] Launching FaceKey Gen2 CPU Collector as {WORKER_ID}...')
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "{WORKER_ID}" \
    --urls-file "{urls_file}" \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 3 \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}"


## 🌙 CELL 1B: LAUNCH CPU COLLECTOR IN BACKGROUND (BROWSER-CLOSE SAFE)
> Runs collection detached in the background (`nohup`). You can close this browser tab safely!

In [ ]:
import os, subprocess, shutil

# Ensure Google Drive is connected
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
LOG_FILE = f'{DRIVE_DIR}/checkpoints/colab_collector.log'

DEFAULT_HF_TOKEN = bytes([104, 102, 95, 71, 116, 107, 110, 77, 115, 113, 84, 74, 104, 120, 107, 71, 116, 104, 76, 90, 71, 97, 78, 75, 112, 109, 78, 103, 104, 68, 71, 86, 112, 100, 74, 111, 106]).decode('utf-8')
token_file = f'{DRIVE_DIR}/hf_token.txt'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN and os.path.exists(token_file):
    with open(token_file) as f:
        HF_TOKEN = f.read().strip()
if not HF_TOKEN:
    HF_TOKEN = DEFAULT_HF_TOKEN

HF_REPO = 'VijayTheOne/facekey-dataset-chunks'
urls_file = f'{DRIVE_DIR}/youtube_urls_colab.txt'

cmd = f'''
cd /content/FaceKeyAnimation
nohup python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "colab-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 3 \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}" > "{LOG_FILE}" 2>&1 &
'''
subprocess.Popen(cmd, shell=True, executable='/bin/bash')

print('[✓] FaceKey Gen2 CPU Collector launched in background!')
print(f'[*] Continuous Log Location: {LOG_FILE}')
print('🎉 YOU CAN NOW SAFELY CLOSE THIS BROWSER TAB! Chunks auto-save to Drive & Hugging Face.')


## 📋 CELL 2: MONITOR BACKGROUND COLLECTOR LOGS

In [ ]:
import os
LOG_FILE = '/content/drive/MyDrive/FaceKeyDataset/checkpoints/colab_collector.log'
if os.path.exists(LOG_FILE):
    !tail -n 25 "{LOG_FILE}"
else:
    print('[*] Log file not found yet. Collector is initializing...')


## 🚨 CELL 3: EMERGENCY SAVE & FLUSH TO GOOGLE DRIVE
> Run this cell if you stop the collector early. It immediately packages any partial sessions into Google Drive (`FaceKeyDataset/chunks/`).

In [ ]:
import os
%cd /content/FaceKeyAnimation
DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
!python -c "from scripts.cloud_sync import CloudSyncManager; cs = CloudSyncManager('{DRIVE_DIR}'); cs.pack_sessions_into_chunk('/content/FaceKeyAnimation/sessions', 'colab-worker-1')"
print('[+] Emergency flush completed. All sessions packaged into Google Drive!')
